In [4]:
import pandas as pd

In [7]:
df = pd.read_csv('Salary Data.csv')

In [9]:
df.head()

,Age,Gender,Education Level,Job Title,Years of Experience,Salary
0,32.0,Male,Bachelor's,Software Engineer,5.0,90000.0
1,28.0,Female,Master's,Data Analyst,3.0,65000.0
2,45.0,Male,PhD,Senior Manager,15.0,150000.0
3,36.0,Female,Bachelor's,Sales Associate,7.0,60000.0
4,52.0,Male,Master's,Director,20.0,200000.0


In [10]:
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 375 entries, 0 to 374
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Age                  373 non-null    float64
 1   Gender               373 non-null    object 
 2   Education Level      373 non-null    object 
 3   Job Title            373 non-null    object 
 4   Years of Experience  373 non-null    float64
 5   Salary               373 non-null    float64
dtypes: float64(3), object(3)
memory usage: 17.7+ KB


Age                    2
Gender                 2
Education Level        2
Job Title              2
Years of Experience    2
Salary                 2
dtype: int64

In [11]:
df = df.dropna()
df.isnull().sum()

Age                    0
Gender                 0
Education Level        0
Job Title              0
Years of Experience    0
Salary                 0
dtype: int64

In [12]:
# Import the tool that converts word-based columns into numbers
from sklearn.preprocessing import LabelEncoder

# Create 3 separate converters — one for each categorical column
# (each column has different word values, so each needs its own encoder)
le_gender = LabelEncoder()
le_education = LabelEncoder()
le_job = LabelEncoder()

# Replace the Gender column's words (Male/Female) with numbers (0/1)
df['Gender'] = le_gender.fit_transform(df['Gender'])

# Replace Education Level's words (Bachelor's/Master's/PhD) with numbers
df['Education Level'] = le_education.fit_transform(df['Education Level'])

# Replace Job Title's words (many different titles) with numbers
df['Job Title'] = le_job.fit_transform(df['Job Title'])

# Preview the table to confirm the word columns are now numbers
df.head()

,Age,Gender,Education Level,Job Title,Years of Experience,Salary
0,32.0,1,0,159,5.0,90000.0
1,28.0,0,1,17,3.0,65000.0
2,45.0,1,2,130,15.0,150000.0
3,36.0,0,0,101,7.0,60000.0
4,52.0,1,1,22,20.0,200000.0


In [ ]:
# X = all the "clue" columns the model will learn from (everything except Salary)
X = df.drop('Salary', axis=1)

# y = the "answer" column what i actually want the model to predict
y = df['Salary']

In [14]:
# Import the tool that splits data into training and testing portions
from sklearn.model_selection import train_test_split

# Split X (clues) and y (answers) into 4 pieces:
# - X_train, y_train = 80% of data, used to TEACH the model
# - X_test, y_test = 20% of data, kept aside to TEST the model on unseen data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% goes into the test set
    random_state=42     # makes the "random" split repeatable every time you run this
)

# Check how many rows ended up in each piece
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 298
Testing rows: 75


In [15]:
# Import the tool that rescales number columns so none of them "dominates" unfairly
from sklearn.preprocessing import StandardScaler

# Create the scaler
scaler = StandardScaler()

# Learn the scale FROM the training data, and apply it to training data
X_train_scaled = scaler.fit_transform(X_train)

# Apply that same scale to the test data (don't relearn from test data that would be cheating)
X_test_scaled = scaler.transform(X_test)

In [16]:
# using Linear Regression from sklearn - starting with the simplest model first
from sklearn.linear_model import LinearRegression

# creating the model, it's empty for now, hasn't learned anything yet
lr = LinearRegression()

# training it on my scaled training data (had to use the scaled version here since Linear Regression needs it)
lr.fit(X_train_scaled, y_train)

# now let's see what it predicts on data it has never seen before (the test set)
pred_lr = lr.predict(X_test_scaled)

# comparing actual salary vs what my model guessed, just to eyeball how close it's getting
comparison = pd.DataFrame({'Actual Salary': y_test, 'Predicted Salary': pred_lr})
comparison.head(10)

,Actual Salary,Predicted Salary
329,180000.0,167633.686952
33,65000.0,93362.332154
15,125000.0,130879.623800
316,80000.0,84293.364421
57,140000.0,161044.594338
240,160000.0,159610.947875
76,160000.0,157000.395059
119,120000.0,104784.305512
307,50000.0,57465.486251
126,95000.0,101175.034515


In [17]:
# importing the tools that measure how good/bad my model's predictions are
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# MAE - on average, how far off are my predictions (in actual salary amount)
mae_lr = mean_absolute_error(y_test, pred_lr)

# RMSE - similar to MAE, but punishes big mistakes harder than small ones
rmse_lr = np.sqrt(mean_squared_error(y_test, pred_lr))

# R2 - what % of the salary differences my model is able to explain (0 to 1, higher is better)
r2_lr = r2_score(y_test, pred_lr)

print(f"MAE: {mae_lr:.2f}")
print(f"RMSE: {rmse_lr:.2f}")
print(f"R2: {r2_lr:.4f}")

MAE: 10928.92
RMSE: 15782.13
R2: 0.8961


In [ ]:
# now trying a Decision Tree: this one can pick up patterns that aren't just a straight line
from sklearn.tree import DecisionTreeRegressor

# setting max_depth=6 so it doesn't go too deep and just memorize the training data
dt = DecisionTreeRegressor(random_state=42, max_depth=6)

# training it - using the plain (unscaled) data this time since trees don't care about scaling
dt.fit(X_train, y_train)

# predicting on the test set
pred_dt = dt.predict(X_test)

# same comparison check as before
comparison_dt = pd.DataFrame({'Actual Salary': y_test, 'Predicted Salary': pred_dt})
comparison_dt.head(10)

In [18]:
# now trying a Decision Tree - this one can pick up patterns that aren't just a straight line
from sklearn.tree import DecisionTreeRegressor

# setting max_depth=6 so it doesn't go too deep and just memorize the training data
dt = DecisionTreeRegressor(random_state=42, max_depth=6)

# training it - using the plain (unscaled) data this time since trees don't care about scaling
dt.fit(X_train, y_train)

# predicting on the test set
pred_dt = dt.predict(X_test)

# same comparison check as before
comparison_dt = pd.DataFrame({'Actual Salary': y_test, 'Predicted Salary': pred_dt})
comparison_dt.head(10)

,Actual Salary,Predicted Salary
329,180000.0,177500.000000
33,65000.0,61666.666667
15,125000.0,112142.857143
316,80000.0,90178.571429
57,140000.0,155312.500000
240,160000.0,155312.500000
76,160000.0,130000.000000
119,120000.0,108939.393939
307,50000.0,50333.333333
126,95000.0,108939.393939


In [19]:
# checking Decision Tree's overall performance across the whole test set, not just these 10 rows
mae_dt = mean_absolute_error(y_test, pred_dt)
rmse_dt = np.sqrt(mean_squared_error(y_test, pred_dt))
r2_dt = r2_score(y_test, pred_dt)

print(f"MAE: {mae_dt:.2f}")
print(f"RMSE: {rmse_dt:.2f}")
print(f"R2: {r2_dt:.4f}")

MAE: 9734.91
RMSE: 13758.02
R2: 0.9211


In [20]:
# comparing how well the tree does on data it TRAINED on vs data it's never seen (test set)
train_r2_dt = r2_score(y_train, dt.predict(X_train))
test_r2_dt = r2_score(y_test, pred_dt)

print(f"Training R2: {train_r2_dt:.4f}")
print(f"Test R2: {test_r2_dt:.4f}")

Training R2: 0.9728
Test R2: 0.9211


In [ ]:
# trying a shallower tree this time to see if it reduces overfitting
dt4 = DecisionTreeRegressor(random_state=42, max_depth=4)

# training on the same data as before
dt4.fit(X_train, y_train)

# predicting on test set
pred_dt4 = dt4.predict(X_test)

# checking both train and test R2 together this time, so i can compare the gap directly
train_r2_dt4 = r2_score(y_train, dt4.predict(X_train))
test_r2_dt4 = r2_score(y_test, pred_dt4)

print(f"Training R2: {train_r2_dt4:.4f}")
print(f"Test R2: {test_r2_dt4:.4f}")
print(f"Gap: {train_r2_dt4 - test_r2_dt4:.4f}")

Training R2: 0.9317
Test R2: 0.9258
Gap: 0.0059


In [22]:
# getting the full picture for our better tree (depth=4), not just R2
mae_dt4 = mean_absolute_error(y_test, pred_dt4)
rmse_dt4 = np.sqrt(mean_squared_error(y_test, pred_dt4))

print(f"MAE: {mae_dt4:.2f}")
print(f"RMSE: {rmse_dt4:.2f}")
print(f"R2: {test_r2_dt4:.4f}")

MAE: 9959.14
RMSE: 13337.33
R2: 0.9258


In [ ]:
# now trying Random Forest - builds 100 trees instead of just 1, and averages their guesses together
from sklearn.ensemble import RandomForestRegressor

# n_estimators=100 means 100 different trees get built
# max_depth=8 limits how deep each individual tree can go (learned from our depth experiment above)
rf = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=8)

# training on the same unscaled data as our trees (Random Forest doesn't need scaling either)
rf.fit(X_train, y_train)

# predicting on the test set
pred_rf = rf.predict(X_test)

# checking train vs test R2 right away this time, since know to look for overfitting from the start
train_r2_rf = r2_score(y_train, rf.predict(X_train))
test_r2_rf = r2_score(y_test, pred_rf)

print(f"Training R2: {train_r2_rf:.4f}")
print(f"Test R2: {test_r2_rf:.4f}")
print(f"Gap: {train_r2_rf - test_r2_rf:.4f}")

Training R2: 0.9829
Test R2: 0.9402
Gap: 0.0428


In [24]:
# testing a shallower Random Forest, same way  tested shallower Decision Trees earlier
rf4 = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=4)
rf4.fit(X_train, y_train)
pred_rf4 = rf4.predict(X_test)

train_r2_rf4 = r2_score(y_train, rf4.predict(X_train))
test_r2_rf4 = r2_score(y_test, pred_rf4)

print(f"max_depth=4 -> Training R2: {train_r2_rf4:.4f}, Test R2: {test_r2_rf4:.4f}, Gap: {train_r2_rf4 - test_r2_rf4:.4f}")

max_depth=4 -> Training R2: 0.9492, Test R2: 0.9268, Gap: 0.0224


In [25]:
rf6 = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=6)
rf6.fit(X_train, y_train)
pred_rf6 = rf6.predict(X_test)

train_r2_rf6 = r2_score(y_train, rf6.predict(X_train))
test_r2_rf6 = r2_score(y_test, pred_rf6)

print(f"max_depth=6 -> Training R2: {train_r2_rf6:.4f}, Test R2: {test_r2_rf6:.4f}, Gap: {train_r2_rf6 - test_r2_rf6:.4f}")

max_depth=6 -> Training R2: 0.9731, Test R2: 0.9377, Gap: 0.0353


In [26]:
# getting the full picture for our chosen Random Forest (depth=6)
mae_rf6 = mean_absolute_error(y_test, pred_rf6)
rmse_rf6 = np.sqrt(mean_squared_error(y_test, pred_rf6))

print(f"MAE: {mae_rf6:.2f}")
print(f"RMSE: {rmse_rf6:.2f}")
print(f"R2: {test_r2_rf6:.4f}")

MAE: 8907.61
RMSE: 12217.76
R2: 0.9377
